# 2. Build the 208-Row Team-Match Dataset

## Purpose

This notebook converts the cleaned match-level dataset into a
team-match dataset for Linear Regression 2.2.

Each of the 104 matches produces two observations:
- One for the listed home team.
- One for the listed away team.

The resulting dataset will contain 208 rows. Each row represents
one team's performance in one match.

The target variable, `goals_scored`, records how many goals that
team scored in the match.

## 2.1 Load the Cleaned Match Data

We load `matches_clean.csv`, which was created in Notebook 1.

This dataset contains one row per match, including the match ID,
stage, date, teams, and goals scored by each side.

In [1]:
import pandas as pd
from pathlib import Path

# Find the project folder whether the notebook runs from the main
# project folder or from the notebooks folder.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Locate the cleaned dataset created in Notebook 1.
input_path = PROJECT_ROOT / "data" / "processed" / "matches_clean.csv"

# Check that the input file exists before loading it.
assert input_path.is_file(), (
    f"File not found: {input_path}. "
    "Run Notebook 1 completely to save matches_clean.csv."
)

# Load the match table and convert match_date to a datetime column.
matches = pd.read_csv(
    input_path,
    parse_dates=["match_date"]
)

display(matches.head())

print("Match-level rows:", len(matches))
print("Unique matches:", matches["match_id"].nunique())

,match_id,stage,match_date,home_team,away_team,home_goals,away_goals
0,1,Group stage,2026-06-11,Mexico,South Africa,2,0
1,2,Group stage,2026-06-11,Korea Republic,Czechia,2,1
2,3,Group stage,2026-06-12,Canada,Bosnia–Herz,1,1
3,4,Group stage,2026-06-12,USA,Paraguay,4,1
4,5,Group stage,2026-06-13,Qatar,Switzerland,1,1


Match-level rows: 104
Unique matches: 104


## 2.2 Create One Observation for Each Team

Each match is converted into two rows.

For the listed home team:
- `team` is the home team.
- `opponent` is the away team.
- `goals_scored` is the home team's score.
- `goals_conceded` is the away team's score.

For the listed away team, these roles are reversed.

The source's home/away labels identify the two sides; they do not
automatically indicate home-ground advantage.

In [2]:
# Create observations from the listed home team's perspective.
home_rows = matches[[
    "match_id",
    "stage",
    "match_date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals"
]].copy()

# Rename the columns to describe a team's perspective consistently.
home_rows.columns = [
    "match_id",
    "stage",
    "match_date",
    "team",
    "opponent",
    "goals_scored",
    "goals_conceded"
]

display(home_rows.head())
print("Home-team observations:", len(home_rows))

,match_id,stage,match_date,team,opponent,goals_scored,goals_conceded
0,1,Group stage,2026-06-11,Mexico,South Africa,2,0
1,2,Group stage,2026-06-11,Korea Republic,Czechia,2,1
2,3,Group stage,2026-06-12,Canada,Bosnia–Herz,1,1
3,4,Group stage,2026-06-12,USA,Paraguay,4,1
4,5,Group stage,2026-06-13,Qatar,Switzerland,1,1


Home-team observations: 104


In [3]:
# Reverse the team and score columns to represent the away team.
away_rows = matches[[
    "match_id",
    "stage",
    "match_date",
    "away_team",
    "home_team",
    "away_goals",
    "home_goals"
]].copy()

# Use exactly the same column names as the home-team table.
away_rows.columns = home_rows.columns

display(away_rows.head())
print("Away-team observations:", len(away_rows))

,match_id,stage,match_date,team,opponent,goals_scored,goals_conceded
0,1,Group stage,2026-06-11,South Africa,Mexico,0,2
1,2,Group stage,2026-06-11,Czechia,Korea Republic,1,2
2,3,Group stage,2026-06-12,Bosnia–Herz,Canada,1,1
3,4,Group stage,2026-06-12,Paraguay,USA,1,4
4,5,Group stage,2026-06-13,Switzerland,Qatar,1,1


Away-team observations: 104


## 2.3 Combine the Two Team Perspectives

We combine both tables to produce 208 observations.

For example, Mexico's 2–0 match against South Africa creates:

| Team | Opponent | Goals scored | Goals conceded |
|---|---|---:|---:|
| Mexico | South Africa | 2 | 0 |
| South Africa | Mexico | 0 | 2 |

Both observations retain the same match ID.

In [4]:
# Stack the home-team and away-team observations.
team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

# Sort chronologically, keeping each match's observations together.
team_matches = team_matches.sort_values(
    ["match_date", "match_id", "team"]
).reset_index(drop=True)

display(team_matches.head(10))

print("Combined team-match observations:", len(team_matches))

,match_id,stage,match_date,team,opponent,goals_scored,goals_conceded
0,1,Group stage,2026-06-11,Mexico,South Africa,2,0
1,1,Group stage,2026-06-11,South Africa,Mexico,0,2
2,2,Group stage,2026-06-11,Czechia,Korea Republic,1,2
3,2,Group stage,2026-06-11,Korea Republic,Czechia,2,1
4,3,Group stage,2026-06-12,Bosnia–Herz,Canada,1,1
5,3,Group stage,2026-06-12,Canada,Bosnia–Herz,1,1
6,4,Group stage,2026-06-12,Paraguay,USA,1,4
7,4,Group stage,2026-06-12,USA,Paraguay,4,1
8,5,Group stage,2026-06-13,Qatar,Switzerland,1,1
9,5,Group stage,2026-06-13,Switzerland,Qatar,1,1


Combined team-match observations: 208


## 2.4 Validate the Team-Match Dataset

We check that:

1. There are 208 observations from 104 matches.
2. Every match contributes exactly two observations.
3. There are 48 distinct teams.
4. No team-match observation is duplicated.
5. Scores are complete, non-negative whole numbers.
6. The two observations for each match correctly reverse the teams
   and scores.

These checks confirm the table's structure and internal consistency.
They do not independently verify the original scores against a source.

In [5]:
# Check the required observation, match, and team counts.
assert len(team_matches) == 208, "Expected 208 team-match rows."
assert team_matches["match_id"].nunique() == 104, "Expected 104 matches."
assert team_matches["team"].nunique() == 48, "Expected 48 teams."

# Check that each match contains two distinct team observations.
assert team_matches.groupby("match_id").size().eq(2).all()
assert not team_matches.duplicated(["match_id", "team"]).any()
assert team_matches["team"].ne(team_matches["opponent"]).all()

# Check that scores are complete, non-negative whole numbers.
score_columns = ["goals_scored", "goals_conceded"]

assert team_matches[score_columns].notna().all().all()
assert team_matches[score_columns].ge(0).all().all()
assert team_matches[score_columns].mod(1).eq(0).all().all()

# Check that the two perspectives for each match mirror each other.
for match_id, pair in team_matches.groupby("match_id"):
    first = pair.iloc[0]
    second = pair.iloc[1]

    assert first["team"] == second["opponent"], (
        f"Team/opponent mismatch in match {match_id}."
    )
    assert first["opponent"] == second["team"], (
        f"Opponent/team mismatch in match {match_id}."
    )
    assert first["goals_scored"] == second["goals_conceded"], (
        f"Score mismatch in match {match_id}."
    )
    assert first["goals_conceded"] == second["goals_scored"], (
        f"Score mismatch in match {match_id}."
    )

print("Team-match rows:", len(team_matches))
print("Unique matches:", team_matches["match_id"].nunique())
print("Unique teams:", team_matches["team"].nunique())
print("Missing score values:", team_matches[score_columns].isna().sum().sum())
print("All structural and score-consistency checks passed.")

Team-match rows: 208
Unique matches: 104
Unique teams: 48
Missing score values: 0
All structural and score-consistency checks passed.


## 2.5 Save the Validated Dataset

We save the validated table as `team_matches_208.csv` in the
`data/processed` folder.

This table contains the target variable and the observed match scores.
The next notebook will calculate eight explanatory variables using
only information available before each match.

In [6]:
# Save the validated dataset for the predictor-building notebook.
output_path = (
    PROJECT_ROOT / "data" / "processed" / "team_matches_208.csv"
)

team_matches.to_csv(output_path, index=False)

print("Saved dataset:", output_path.name)
print("Dataset dimensions:", team_matches.shape)

Saved dataset: team_matches_208.csv
Dataset dimensions: (208, 7)
